# Experiment 3A: Correlation

**Aim:** Given a data set of 10 rows, calculate Karl Pearson's coefficient of correlation and Spearman's rank correlation coefficient (using repeated ranks) manually. Then write a Python program to calculate both coefficients and match with manually calculated values. Solve the real world problem statements.

## Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
X = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
Y = np.array([5, 6, 7, 8, 7, 9, 10, 10, 11, 12])

df = pd.DataFrame({'X': X, 'Y': Y})
df

## Scatter Plot

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X, Y, color='steelblue', edgecolors='black', s=80, zorder=5)
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Scatter Plot of X vs Y')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 1. Karl Pearson's Correlation Coefficient

$$r = \frac{n\sum XY - \sum X \sum Y}{\sqrt{[n\sum X^2 - (\sum X)^2][n\sum Y^2 - (\sum Y)^2]}}$$

### Manual Calculation (Step-by-Step)

In [ ]:
n = len(X)

manual = pd.DataFrame({
    'X': X,
    'Y': Y,
    'X²': X**2,
    'Y²': Y**2,
    'XY': X * Y
})

manual.loc['Sum'] = manual.sum()
manual

In [ ]:
sum_X = np.sum(X)
sum_Y = np.sum(Y)
sum_X2 = np.sum(X**2)
sum_Y2 = np.sum(Y**2)
sum_XY = np.sum(X * Y)

print(f"n       = {n}")
print(f"ΣX      = {sum_X}")
print(f"ΣY      = {sum_Y}")
print(f"ΣX²     = {sum_X2}")
print(f"ΣY²     = {sum_Y2}")
print(f"ΣXY     = {sum_XY}")

numerator = n * sum_XY - sum_X * sum_Y
denominator = np.sqrt((n * sum_X2 - sum_X**2) * (n * sum_Y2 - sum_Y**2))

r_pearson_manual = numerator / denominator

print(f"\nNumerator   = {n}×{sum_XY} - {sum_X}×{sum_Y} = {numerator}")
print(f"Denominator = √[({n}×{sum_X2} - {sum_X}²)({n}×{sum_Y2} - {sum_Y}²)]")
print(f"            = √[{n*sum_X2 - sum_X**2} × {n*sum_Y2 - sum_Y**2}]")
print(f"            = {denominator:.4f}")
print(f"\nKarl Pearson's r (manual) = {r_pearson_manual:.4f}")

### Python Function for Pearson Correlation

In [ ]:
def pearson_correlation(X, Y):
    n = len(X)
    sum_X = np.sum(X)
    sum_Y = np.sum(Y)
    sum_X2 = np.sum(X**2)
    sum_Y2 = np.sum(Y**2)
    sum_XY = np.sum(X * Y)

    numerator = n * sum_XY - sum_X * sum_Y
    denominator = np.sqrt((n * sum_X2 - sum_X**2) * (n * sum_Y2 - sum_Y**2))

    return numerator / denominator

r_pearson = pearson_correlation(X, Y)
print(f"Karl Pearson's r (function) = {r_pearson:.4f}")

## 2. Spearman's Rank Correlation Coefficient (with Repeated Ranks)

When there are **no ties**, the formula is:

$$r_s = 1 - \frac{6 \sum d_i^2}{n(n^2 - 1)}$$

When there are **tied (repeated) ranks**, we use the corrected formula:

$$r_s = \frac{A + B - \sum d_i^2}{2\sqrt{A \cdot B}}$$

where:
- $A = \frac{n^3 - n}{12} - \sum T_X$, $\quad B = \frac{n^3 - n}{12} - \sum T_Y$
- $T = \frac{t^3 - t}{12}$ for each group of $t$ tied values

### Manual Calculation (Step-by-Step)

In [ ]:
from scipy.stats import rankdata

rank_X = rankdata(X, method='average')
rank_Y = rankdata(Y, method='average')
d = rank_X - rank_Y
d2 = d**2

rank_table = pd.DataFrame({
    'X': X,
    'Y': Y,
    'Rank_X': rank_X,
    'Rank_Y': rank_Y,
    'd = Rx - Ry': d,
    'd²': d2
})

rank_table.loc['Sum'] = rank_table.sum()
rank_table

In [ ]:
sum_d2 = np.sum(d2)
print(f"Σd² = {sum_d2}")

print("\nChecking for repeated (tied) ranks:")

def tie_correction(values):
    unique, counts = np.unique(values, return_counts=True)
    T = 0
    for val, cnt in zip(unique, counts):
        if cnt > 1:
            t_val = (cnt**3 - cnt) / 12
            print(f"  Value {val} appears {cnt} times → T = ({cnt}³ - {cnt})/12 = {t_val:.2f}")
            T += t_val
    return T

print("\nIn X:")
Tx = tie_correction(X)
if Tx == 0:
    print("  No ties in X")

print("\nIn Y:")
Ty = tie_correction(Y)

base = (n**3 - n) / 12
A = base - Tx
B = base - Ty

print(f"\nBase = (n³ - n)/12 = ({n}³ - {n})/12 = {base:.2f}")
print(f"A = {base:.2f} - {Tx:.2f} = {A:.2f}")
print(f"B = {base:.2f} - {Ty:.2f} = {B:.2f}")

r_spearman_manual = (A + B - sum_d2) / (2 * np.sqrt(A * B))

print(f"\nr_s = (A + B - Σd²) / (2√(A·B))")
print(f"    = ({A:.2f} + {B:.2f} - {sum_d2}) / (2 × √({A:.2f} × {B:.2f}))")
print(f"    = {A + B - sum_d2:.2f} / {2 * np.sqrt(A * B):.4f}")
print(f"    = {r_spearman_manual:.4f}")
print(f"\nSpearman's r_s (manual) = {r_spearman_manual:.4f}")

### Python Function for Spearman Rank Correlation

In [ ]:
def spearman_rank_correlation(X, Y):
    n = len(X)
    rank_X = rankdata(X, method='average')
    rank_Y = rankdata(Y, method='average')
    d = rank_X - rank_Y
    sum_d2 = np.sum(d**2)

    def tc(vals):
        _, counts = np.unique(vals, return_counts=True)
        return sum((t**3 - t) / 12 for t in counts if t > 1)

    base = (n**3 - n) / 12
    A = base - tc(X)
    B = base - tc(Y)
    r_s = (A + B - sum_d2) / (2 * np.sqrt(A * B))
    return r_s

r_spearman = spearman_rank_correlation(X, Y)
print(f"Spearman's r_s (function) = {r_spearman:.4f}")

## 3. Verify Spearman's Rank using SciPy

In [ ]:
scipy_result = stats.spearmanr(X, Y)
print(f"Spearman's r_s (scipy)    = {scipy_result.statistic:.4f}")
print(f"p-value                   = {scipy_result.pvalue:.4f}")

## 4. Summary — All Three Results

In [ ]:
print("=" * 55)
print(f"{'Method':<35} {'Value':>10}")
print("=" * 55)
print(f"{'Karl Pearson r (manual)':<35} {r_pearson_manual:>10.4f}")
print(f"{'Karl Pearson r (function)':<35} {r_pearson:>10.4f}")
print("-" * 55)
print(f"{'Spearman r_s (manual)':<35} {r_spearman_manual:>10.4f}")
print(f"{'Spearman r_s (function)':<35} {r_spearman:>10.4f}")
print(f"{'Spearman r_s (scipy)':<35} {scipy_result.statistic:>10.4f}")
print("=" * 55)

print(f"\nPearson match:  {'✓ YES' if round(r_pearson_manual, 4) == round(r_pearson, 4) else '✗ NO'}")
print(f"Spearman match: {'✓ YES' if round(r_spearman_manual, 4) == round(scipy_result.statistic, 4) else '✗ NO'}")

---
## Real World Problem 1

**Q1.** The following table gives the data on weekly family consumption expenditure (Y) and weekly family income (X).

| Y | 70 | 65 | 90 | 95 | 110 | 115 | 120 | 140 | 155 | 150 |
|---|---|---|---|---|---|---|---|---|---|---|
| **X** | 80 | 100 | 120 | 140 | 160 | 180 | 200 | 220 | 240 | 260 |

**(i)** Compute the coefficient of correlation between X and Y.

**(ii)** Test the significance of the coefficient of correlation at 5% level of significance.

In [ ]:
X1 = np.array([80, 100, 120, 140, 160, 180, 200, 220, 240, 260])
Y1 = np.array([70, 65, 90, 95, 110, 115, 120, 140, 155, 150])

df_rwp1 = pd.DataFrame({'X (Income)': X1, 'Y (Expenditure)': Y1})
df_rwp1

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X1, Y1, color='coral', edgecolors='black', s=80, zorder=5)
plt.xlabel('Weekly Family Income (X)')
plt.ylabel('Weekly Family Consumption Expenditure (Y)')
plt.title('RWP 1: Income vs Expenditure')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### (i) Coefficient of Correlation

In [ ]:
r_p1 = pearson_correlation(X1, Y1)
r_s1 = spearman_rank_correlation(X1, Y1)
scipy_s1 = stats.spearmanr(X1, Y1)

print(f"(i) Karl Pearson's r       = {r_p1:.4f}")
print(f"    Spearman's r_s (func)  = {r_s1:.4f}")
print(f"    Spearman's r_s (scipy) = {scipy_s1.statistic:.4f}")

### (ii) Test of Significance at 5% level

$$t = \frac{r\sqrt{n-2}}{\sqrt{1-r^2}}$$

Degrees of freedom: $df = n - 2$. Compare with $t_{\alpha/2, df}$ (two-tailed).

In [ ]:
n1 = len(X1)
df1 = n1 - 2
t_stat1 = (r_p1 * np.sqrt(df1)) / np.sqrt(1 - r_p1**2)
t_critical1 = stats.t.ppf(1 - 0.05/2, df1)
p_value1 = 2 * (1 - stats.t.cdf(abs(t_stat1), df1))

print(f"n = {n1}, df = {df1}")
print(f"t-statistic = {t_stat1:.4f}")
print(f"t-critical (α=0.05, two-tailed) = ±{t_critical1:.4f}")
print(f"p-value = {p_value1:.6f}")
print()
if abs(t_stat1) > t_critical1:
    print(f"Since |t| = {abs(t_stat1):.4f} > t_critical = {t_critical1:.4f},")
    print("we REJECT H₀. The correlation is statistically SIGNIFICANT at 5% level.")
else:
    print(f"Since |t| = {abs(t_stat1):.4f} ≤ t_critical = {t_critical1:.4f},")
    print("we FAIL TO REJECT H₀. The correlation is NOT significant at 5% level.")

---
## Real World Problem 2

**Q2.** The following table gives the per capita household expenditure on food (Y) and per capita total household expenditure (X).

| Y | 60 | 90 | 110 | 125 | 150 | 170 | 180 | 200 | 220 | 230 | 240 | 250 | 255 | 260 | 260 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| **X** | 100 | 150 | 200 | 250 | 300 | 350 | 400 | 450 | 500 | 550 | 600 | 650 | 700 | 750 | 800 |

**(i)** Compute the coefficient of correlation between X and Y.

**(ii)** Test the significance of the coefficient of correlation at 5% level of significance.

In [ ]:
X2 = np.array([100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800])
Y2 = np.array([60, 90, 110, 125, 150, 170, 180, 200, 220, 230, 240, 250, 255, 260, 260])

df_rwp2 = pd.DataFrame({'X (Total Expenditure)': X2, 'Y (Food Expenditure)': Y2})
df_rwp2

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X2, Y2, color='teal', edgecolors='black', s=80, zorder=5)
plt.xlabel('Per Capita Total Household Expenditure (X)')
plt.ylabel('Per Capita Food Expenditure (Y)')
plt.title('RWP 2: Total Expenditure vs Food Expenditure')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### (i) Coefficient of Correlation

In [ ]:
r_p2 = pearson_correlation(X2, Y2)
r_s2 = spearman_rank_correlation(X2, Y2)
scipy_s2 = stats.spearmanr(X2, Y2)

print(f"(i) Karl Pearson's r       = {r_p2:.4f}")
print(f"    Spearman's r_s (func)  = {r_s2:.4f}")
print(f"    Spearman's r_s (scipy) = {scipy_s2.statistic:.4f}")

### (ii) Test of Significance at 5% level

In [ ]:
n2 = len(X2)
df2 = n2 - 2
t_stat2 = (r_p2 * np.sqrt(df2)) / np.sqrt(1 - r_p2**2)
t_critical2 = stats.t.ppf(1 - 0.05/2, df2)
p_value2 = 2 * (1 - stats.t.cdf(abs(t_stat2), df2))

print(f"n = {n2}, df = {df2}")
print(f"t-statistic = {t_stat2:.4f}")
print(f"t-critical (α=0.05, two-tailed) = ±{t_critical2:.4f}")
print(f"p-value = {p_value2:.6f}")
print()
if abs(t_stat2) > t_critical2:
    print(f"Since |t| = {abs(t_stat2):.4f} > t_critical = {t_critical2:.4f},")
    print("we REJECT H₀. The correlation is statistically SIGNIFICANT at 5% level.")
else:
    print(f"Since |t| = {abs(t_stat2):.4f} ≤ t_critical = {t_critical2:.4f},")
    print("we FAIL TO REJECT H₀. The correlation is NOT significant at 5% level.")

## Conclusion

We have calculated Karl Pearson's coefficient of correlation and Spearman's rank correlation coefficient (using repeated ranks) manually and implemented Python functions to calculate both. Both the coefficients match with the manually computed values and the SciPy verification.

**Real World Problem 1:** Weekly family income and consumption expenditure show a strong positive correlation that is statistically significant at 5% level.

**Real World Problem 2:** Per capita total household expenditure and food expenditure show a strong positive correlation that is statistically significant at 5% level.